# RAG con LangChain: Implementación y Experimentos

Este cuaderno implementa un sistema RAG (Retrieval-Augmented Generation) de extremo a extremo usando LangChain con:
- Carga de documentos (community loaders)
- Separadores de texto (text splitters)
- Embeddings y LLMs con Ollama
- Base vectorial Chroma con persistencia
- LLMs de Google GenAI (Gemini)
- Experimentos de embeddings/LLMs, optimización de chunking, gestión avanzada de Chroma y refinamiento de prompts

Documento X de ejemplo: `data/documento_x.txt` (incluido en este proyecto).

Ejecute las celdas en orden. Si usa Ollama o Gemini, revise las notas de configuración.


In [ ]:
# Instalación de dependencias (ejecutar una vez)
%pip install -U -q \
    langchain langchain-community langchain-text-splitters \
    langchain-ollama langchain-chroma langchain-google-genai \
    chromadb pypdf python-dotenv tqdm matplotlib pandas \
    faiss-cpu qdrant-client

## Configuración de entorno
- `GOOGLE_API_KEY`: clave para usar Gemini vía `langchain_google_genai`.
- `OLLAMA_HOST`: URL de servidor Ollama si no es local (por defecto http://127.0.0.1:11434).

Opciones:
1) Crear un archivo `.env` en la raíz del proyecto:
```
GOOGLE_API_KEY=tu_clave
OLLAMA_HOST=http://127.0.0.1:11434
```
2) O bien exportar variables de entorno en su sistema.

Notas:
- Para usar Ollama necesita tener instalado Ollama y haber descargado los modelos: `ollama pull llama3.1` y embeddings como `ollama pull nomic-embed-text`.
- Para usar Gemini necesita una clave válida de Google AI Studio.


In [ ]:
# Importaciones clave y utilidades
import os, json, time, math
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any

from dotenv import load_dotenv

# LangChain - loaders y splitters
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
)

# LangChain - embeddings y LLMs
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain - Core (prompts, runnables y parsing)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Vector stores alternativos
from langchain_community.vectorstores import FAISS
try:
    from langchain_community.vectorstores import Qdrant
    import qdrant_client
except Exception:
    Qdrant = None

# Utilidades

def timer():
    class _T:
        def __enter__(self):
            self.t0 = time.perf_counter()
            return self
        def __exit__(self, exc_type, exc, tb):
            self.dt = time.perf_counter() - self.t0
    return _T()


def print_sources(docs: List[Any]):
    uniq = []
    seen = set()
    for d in docs:
        meta = d.metadata or {}
        src = meta.get("source", "?")
        page = meta.get("page", meta.get("page_number", "?"))
        key = (src, page)
        if key not in seen:
            seen.add(key)
            uniq.append(f"- {src} (página {page})")
    if not uniq:
        print("(sin fuentes)")
    else:
        print("Fuentes:\n" + "\n".join(uniq))


def format_docs(docs: List[Any]) -> str:
    blocks = []
    for i, d in enumerate(docs, 1):
        meta = d.metadata or {}
        src = meta.get("source", "?")
        page = meta.get("page", meta.get("page_number", "?"))
        blocks.append(f"[Fragmento {i} | fuente: {src} | página: {page}]\n{d.page_content}")
    return "\n\n".join(blocks)

In [ ]:
# Definir rutas, modelos y variables de entorno
load_dotenv()

DOC_PATH = "data/documento_x.txt"  # Cambiar a .pdf si corresponde
PERSIST_DIR = ".chroma/docs_x"
COLLECTION_NAME = "docs_x"

# Modelos por defecto
EMB_MODEL = "nomic-embed-text"  # Alternativas: mxbai-embed-large, snowflake-arctic-embed, bge-m3, all-minilm
LLM_MODEL_OLLAMA = "llama3.1"    # Alternativas: mistral, qwen2, llama3
LLM_MODEL_GEMINI = "gemini-1.5-flash"

# Claves/API
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY:
    print("[Aviso] GOOGLE_API_KEY no configurada. Las celdas de Gemini se omitirán.")

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")
os.environ.setdefault("OLLAMA_HOST", OLLAMA_HOST)
print("OLLAMA_HOST=", os.environ.get("OLLAMA_HOST"))

# Verificar documento
if not Path(DOC_PATH).exists():
    raise FileNotFoundError(f"No existe el documento: {DOC_PATH}. Añádalo antes de continuar.")

## Carga de documento X con document_loaders
Usamos `TextLoader` para `.txt`/`.md` y `PyPDFLoader` para `.pdf`. Se mostrarán metadatos básicos.

In [ ]:
# Cargar el documento
ext = Path(DOC_PATH).suffix.lower()
if ext == ".pdf":
    loader = PyPDFLoader(DOC_PATH)
else:
    loader = TextLoader(DOC_PATH, autodetect_encoding=True)

docs = loader.load()
print(f"Documentos cargados: {len(docs)}")
if docs:
    print("Ejemplo de metadatos:", docs[0].metadata)

## Enriquecimiento de metadatos del documento
Normalizamos y añadimos metadatos útiles (source, page, type, timestamp) para filtrado y trazabilidad.

In [ ]:
# Enriquecer metadatos
now = datetime.utcnow().isoformat()
for d in docs:
    meta = d.metadata or {}
    meta["source"] = meta.get("source", Path(DOC_PATH).name)
    if "page" not in meta and "page_number" not in meta:
        meta["page"] = meta.get("page", 1)
    meta["type"] = Path(DOC_PATH).suffix.lower().lstrip(".")
    meta["timestamp"] = now
    d.metadata = meta

print("Metadatos ejemplo tras normalización:", docs[0].metadata if docs else {})

## División de texto con CharacterTextSplitter (línea base)
Parámetros iniciales: `chunk_size=800`, `chunk_overlap=100`, `separator="\n\n"`.

In [ ]:
# División en chunks (baseline)
chunk_size = 800
chunk_overlap = 100
splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separator="\n\n",
)
chunks = splitter.split_documents(docs)

lengths = [len(c.page_content) for c in chunks]
print(f"Chunks creados: {len(chunks)} | Tamaño medio: {sum(lengths)//max(1,len(lengths))} chars")
print("Ejemplo de chunk:\n", chunks[0].page_content[:300] + ("..." if len(chunks[0].page_content)>300 else ""))

## Construcción de embeddings con OllamaEmbeddings (línea base)
Inicializamos `OllamaEmbeddings` y verificamos latencia con `embed_query("ping")`. Asegúrese de tener el modelo de embeddings descargado en Ollama.

In [ ]:
# Inicializar embeddings y prueba de latencia
try:
    with timer() as t:
        embeddings = OllamaEmbeddings(model=EMB_MODEL)
        _ = embeddings.embed_query("ping")
    print(f"Embeddings OK: {EMB_MODEL} | Latencia embed_query ~ {t.dt:.3f}s")
except Exception as e:
    print("[Error] No se pudo inicializar embeddings con Ollama.")
    print("Causas comunes: Ollama no está corriendo o el modelo no está descargado.")
    print("Sugerencia: instale/arranque Ollama y ejecute: ollama pull", EMB_MODEL)
    raise

## Creación y persistencia de Chroma (vector store)
Creamos la colección en `PERSIST_DIR` con nombre `COLLECTION_NAME`, insertamos los chunks y persistimos.

In [ ]:
# Crear/Persistir Chroma
from shutil import rmtree
Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)

with timer() as t_idx:
    vs = Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIR,
        embedding_function=embeddings,
    )
    # Limpiamos colección previa con el mismo nombre para resultados reproducibles
    try:
        existing = vs.get()
        if existing and existing.get("ids"):
            vs.delete(existing["ids"])  # borrar todo
    except Exception:
        pass

    vs.add_documents(chunks)
    vs.persist()

col_info = vs.get()
print(f"Indexado listo en {t_idx.dt:.2f}s | Total embeddings: {len(col_info.get('ids', []))}")

## Construcción del recuperador (similaridad y MMR)
Creamos un retriever por similaridad y opcionalmente por MMR (diversidad controlada).

In [ ]:
# Retrievers
retriever_sim = vs.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever_mmr = vs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 6, "fetch_k": 20, "lambda_mult": 0.5},
)

# Prueba rápida de recuperación
query_demo = "¿Qué es RAG?"
with timer() as t_ret:
    docs_demo = retriever_sim.get_relevant_documents(query_demo)
print(f"Recuperación (similarity) ~ {t_ret.dt:.3f}s | docs: {len(docs_demo)}")
print_sources(docs_demo)

In [ ]:
# (Ajuste) Import adicional para LCEL
from langchain_core.runnables import RunnableLambda

## Configuración de LLMs: ChatOllama y ChatGoogleGenerativeAI
Instanciamos LLMs locales (Ollama) y en la nube (Gemini) para comparar latencia y calidad.

In [ ]:
# Instanciar LLMs y prueba de conectividad
llm_ollama = None
llm_gemini = None

try:
    with timer() as t_llm1:
        llm_ollama = ChatOllama(model=LLM_MODEL_OLLAMA, temperature=0.2)
        _ = llm_ollama.invoke("Di 'hola' en una palabra.")
    print(f"Ollama OK: {LLM_MODEL_OLLAMA} | Latencia ~ {t_llm1.dt:.2f}s")
except Exception as e:
    print("[Aviso] No se pudo inicializar ChatOllama:", e)
    print("Verifique que Ollama esté activo y el modelo descargado (ollama pull)")

if GOOGLE_API_KEY:
    try:
        with timer() as t_llm2:
            llm_gemini = ChatGoogleGenerativeAI(
                model=LLM_MODEL_GEMINI,
                temperature=0.2,
                google_api_key=GOOGLE_API_KEY,
            )
            _ = llm_gemini.invoke("Di 'hola' en una palabra.")
        print(f"Gemini OK: {LLM_MODEL_GEMINI} | Latencia ~ {t_llm2.dt:.2f}s")
    except Exception as e:
        print("[Aviso] No se pudo inicializar Gemini:", e)
else:
    print("[Nota] GOOGLE_API_KEY no establecido: se omitirá Gemini en las pruebas.")

## Template de prompt base con citas de fuentes
El prompt instruye al LLM a usar el contexto recuperado, responder en español y citar fuentes con `source` y `page` en una sección "Fuentes:".

In [ ]:
# Definir prompt base
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Eres un asistente experto. Responde en español. "
     "Usa exclusivamente el CONTEXTO proporcionado. Si la respuesta no está en el contexto, di que no hay evidencia suficiente. "
     "Incluye una sección 'Fuentes:' al final, listando 'source' y 'page' de los fragmentos usados."),
    ("human", "Pregunta: {question}\n\nCONTEXTO:\n{context}\n\nResponde de forma concisa y precisa.")
])
parser = StrOutputParser()

## Encadenamiento RAG con LCEL y función `ask`
El chain combina: `retriever` → `prompt` → `llm` → parser. La función `ask` devuelve respuesta y fuentes utilizadas.

In [ ]:
# Construir chain RAG y helper ask()
def build_rag_chain(retriever, llm):
    chain = (
        {"context": retriever | RunnableLambda(format_docs),
         "question": RunnablePassthrough()}
        | prompt
        | llm
        | parser
    )
    return chain


def ask(question: str, llm, retriever):
    chain = build_rag_chain(retriever, llm)
    with timer() as t:
        answer = chain.invoke(question)
    # Recuperamos docs para fuentes
    docs_used = retriever.get_relevant_documents(question)
    print(f"Tiempo total: {t.dt:.2f}s")
    print_sources(docs_used)
    return answer, docs_used

# Prueba rápida (si hay LLM activo)
if llm_ollama is not None:
    ans, srcs = ask("¿Qué es RAG y cuáles son sus componentes?", llm_ollama, retriever_sim)
    print("\nRespuesta (Ollama):\n", ans[:1000])
elif llm_gemini is not None:
    ans, srcs = ask("¿Qué es RAG y cuáles son sus componentes?", llm_gemini, retriever_sim)
    print("\nRespuesta (Gemini):\n", ans[:1000])
else:
    print("[Nota] Ningún LLM disponible para la prueba rápida.")

## Verificación funcional: preguntas de prueba y evaluación simple
Definimos preguntas sobre el documento y una métrica sencilla: detección de palabras clave y existencia de fuentes.

In [ ]:
# Preguntas y evaluación simple
questions = [
    "¿Qué es RAG y cuáles son sus componentes?",
    "¿Por qué es importante el solapamiento entre chunks?",
    "¿Cuándo elegir Chroma frente a otras bases vectoriales?",
    "¿Qué prácticas ayudan a mejorar la atribución de fuentes?",
]

expected_keywords = {
    questions[0]: ["RAG", "componentes", "documentos", "LLM"],
    questions[1]: ["solap", "contexto"],
    questions[2]: ["Chroma", "persistencia"],
    questions[3]: ["fuentes", "cita"],
}


def evaluate(answer: str, docs: list, expected_terms: list[str]):
    ans_l = answer.lower() if answer else ""
    kw_ok = all(any(term.lower() in ans_l for term in [kw, kw.replace("á", "a")]) for kw in expected_terms)
    src_ok = len(docs) > 0
    score = int(kw_ok) + int(src_ok)
    return {"kw_ok": kw_ok, "src_ok": src_ok, "score_0_2": score}


if llm_ollama is not None:
    results = []
    for q in questions:
        print("\n=== Pregunta:", q)
        ans, srcs = ask(q, llm_ollama, retriever_sim)
        print("Respuesta:\n", ans[:1200])
        res = evaluate(ans, srcs, expected_keywords.get(q, []))
        print("Métrica simple:", res)
        results.append({"q": q, **res})
    try:
        import pandas as pd
        display(pd.DataFrame(results))
    except Exception:
        print(results)
else:
    print("[Nota] Saltando evaluación: no hay LLM Ollama disponible.")

## Experimentos con modelos de embedding alternativos (Ollama)
Probamos varios modelos de embeddings y comparamos latencia de indexado/recuperación y una métrica simple de precisión.

In [ ]:
# Benchmark de embeddings (requiere tener descargados los modelos en Ollama)
embed_models = [
    "nomic-embed-text",
    "mxbai-embed-large",
    "snowflake-arctic-embed",
    "bge-m3",
    "all-minilm",
]

bench_rows = []

for model in embed_models:
    print(f"\n### Embeddings: {model}")
    try:
        emb = OllamaEmbeddings(model=model)
        with timer() as t_e:
            _ = emb.embed_query("ping")
        # Re-crear colección temporal para este embedding
        tmp_dir = f".chroma/emb_{model.replace('/', '_')}"
        coll = f"docs_x_{model[:20]}"
        vs_tmp = Chroma(collection_name=coll, persist_directory=tmp_dir, embedding_function=emb)
        # limpiar
        try:
            ex = vs_tmp.get()
            if ex and ex.get("ids"):
                vs_tmp.delete(ex["ids"])
        except Exception:
            pass
        with timer() as t_idx:
            vs_tmp.add_documents(chunks)
            vs_tmp.persist()
        # Retrieval rápida con 1 pregunta
        ret = vs_tmp.as_retriever(search_type="similarity", search_kwargs={"k": 4})
        with timer() as t_r:
            docs_r = ret.get_relevant_documents(questions[0])
        # Evaluación simple (usa llm_ollama si disponible)
        score = None
        if llm_ollama is not None:
            ans, srcs = ask(questions[0], llm_ollama, ret)
            score = evaluate(ans, srcs, expected_keywords.get(questions[0], [])).get("score_0_2")
        bench_rows.append({
            "model": model,
            "embed_latency_s": round(t_e.dt, 3),
            "index_latency_s": round(t_idx.dt, 3),
            "retrieval_latency_s": round(t_r.dt, 3),
            "retrieved_docs": len(docs_r),
            "score_0_2": score,
        })
    except Exception as e:
        print("[Error]", e)

try:
    import pandas as pd
    df = pd.DataFrame(bench_rows)
    display(df.sort_values(by=["score_0_2", "retrieval_latency_s"], ascending=[False, True]))
except Exception:
    print(bench_rows)

## Comparación de LLMs (Ollama vs Google GenAI)
Probamos varios modelos manteniendo el mismo vector store para aislar el efecto del LLM.

In [ ]:
# Benchmark de LLMs
ollama_models = ["llama3.1", "mistral", "qwen2"]
gemini_models = ["gemini-1.5-flash", "gemini-1.5-pro"] if GOOGLE_API_KEY else []

llm_rows = []

for m in ollama_models:
    try:
        print(f"\n### LLM (Ollama): {m}")
        with timer() as t:
            llm = ChatOllama(model=m, temperature=0.2)
            ans, srcs = ask(questions[0], llm, retriever_sim)
        score = evaluate(ans, srcs, expected_keywords.get(questions[0], [])).get("score_0_2")
        llm_rows.append({"provider": "ollama", "model": m, "latency_s": round(t.dt,2), "score_0_2": score})
    except Exception as e:
        print("[Aviso]", e)

for m in gemini_models:
    try:
        print(f"\n### LLM (Gemini): {m}")
        with timer() as t:
            llm = ChatGoogleGenerativeAI(model=m, temperature=0.2, google_api_key=GOOGLE_API_KEY)
            ans, srcs = ask(questions[0], llm, retriever_sim)
        score = evaluate(ans, srcs, expected_keywords.get(questions[0], [])).get("score_0_2")
        llm_rows.append({"provider": "gemini", "model": m, "latency_s": round(t.dt,2), "score_0_2": score})
    except Exception as e:
        print("[Aviso]", e)

try:
    import pandas as pd
    display(pd.DataFrame(llm_rows).sort_values(by=["score_0_2", "latency_s"], ascending=[False, True]))
except Exception:
    print(llm_rows)

## Optimización de chunk_size y chunk_overlap (búsqueda en malla)
Probamos combinaciones de parámetros y medimos precisión simple y latencias.

In [ ]:
# Grid search de chunking
cs_list = [300, 600, 900]
co_list = [50, 120, 200]

chunk_rows = []

for cs in cs_list:
    for co in co_list:
        print(f"\n### chunk_size={cs} chunk_overlap={co}")
        sp = CharacterTextSplitter(chunk_size=cs, chunk_overlap=co, separator="\n\n")
        chunks_try = sp.split_documents(docs)
        try:
            emb = OllamaEmbeddings(model=EMB_MODEL)
            tmp_dir = f".chroma/chunk_cs{cs}_co{co}"
            coll = f"docs_x_cs{cs}_co{co}"
            vs_tmp = Chroma(collection_name=coll, persist_directory=tmp_dir, embedding_function=emb)
            # limpiar
            try:
                ex = vs_tmp.get()
                if ex and ex.get("ids"):
                    vs_tmp.delete(ex["ids"])
            except Exception:
                pass
            with timer() as t_idx:
                vs_tmp.add_documents(chunks_try)
                vs_tmp.persist()
            ret = vs_tmp.as_retriever(search_type="similarity", search_kwargs={"k": 4})
            with timer() as t_r:
                docs_r = ret.get_relevant_documents(questions[0])
            score = None
            if llm_ollama is not None:
                ans, srcs = ask(questions[0], llm_ollama, ret)
                score = evaluate(ans, srcs, expected_keywords.get(questions[0], [])).get("score_0_2")
            chunk_rows.append({
                "chunk_size": cs,
                "chunk_overlap": co,
                "n_chunks": len(chunks_try),
                "index_latency_s": round(t_idx.dt, 2),
                "retrieval_latency_s": round(t_r.dt, 2),
                "score_0_2": score,
            })
        except Exception as e:
            print("[Aviso]", e)

try:
    import pandas as pd
    df_g = pd.DataFrame(chunk_rows)
    display(df_g.sort_values(by=["score_0_2", "retrieval_latency_s"], ascending=[False, True]))
except Exception:
    print(chunk_rows)

## Otros separadores de texto (Recursive, Markdown)
Probamos `RecursiveCharacterTextSplitter` y `MarkdownHeaderTextSplitter` para ver si preservan mejor la estructura.

In [ ]:
# Splitters alternativos
rec_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)
chunks_rec = rec_splitter.split_documents(docs)
print("Recursive -> n_chunks:", len(chunks_rec))

if DOC_PATH.lower().endswith((".md",)):
    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")])
    md_docs = md_splitter.split_text(Path(DOC_PATH).read_text(encoding="utf-8"))
    print("MarkdownHeader -> n_docs:", len(md_docs))
else:
    print("MarkdownHeaderTextSplitter omitido (el documento no es .md)")

# Prueba de retrieval con recursive
try:
    emb = OllamaEmbeddings(model=EMB_MODEL)
    vs_rec = Chroma(collection_name="docs_x_recursive", persist_directory=".chroma/rec", embedding_function=emb)
    try:
        ex = vs_rec.get();
        if ex and ex.get("ids"): vs_rec.delete(ex["ids"])
    except Exception:
        pass
    vs_rec.add_documents(chunks_rec)
    vs_rec.persist()
    ret_rec = vs_rec.as_retriever(search_type="similarity", search_kwargs={"k": 4})
    ans, srcs = ask(questions[0], llm_ollama if llm_ollama else (llm_gemini if llm_gemini else None), ret_rec) if (llm_ollama or llm_gemini) else (None, [])
except Exception as e:
    print("[Aviso]", e)

## Funcionalidades avanzadas de Chroma: filtros, MMR, actualización y persistencia
Mostramos filtrado por metadatos, modos de búsqueda (similaridad, MMR, umbral de score), actualización/borrado y recarga de la colección desde disco.

In [ ]:
# Filtros de metadatos y modos de búsqueda
base_source = Path(DOC_PATH).name

# Similaridad con filtro por 'source'
docs_f = vs.similarity_search_with_relevance_scores(
    "Define RAG en una frase",
    k=4,
    filter={"source": base_source},
)
print("Filtrado por source -> resultados:", len(docs_f))

# Similaridad con umbral de score (si compatible)
try:
    ret_thr = vs.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 6, "score_threshold": 0.2},
    )
    docs_thr = ret_thr.get_relevant_documents("componentes de RAG")
    print("Threshold -> resultados:", len(docs_thr))
except Exception as e:
    print("[Aviso] similarity_score_threshold no disponible:", e)

# MMR
ret_mmr = vs.as_retriever(search_type="mmr", search_kwargs={"k": 6, "fetch_k": 20, "lambda_mult": 0.5})
docs_mmr = ret_mmr.get_relevant_documents("componentes de RAG")
print("MMR -> resultados:", len(docs_mmr))

# Actualización/borrado de documentos (estrategia: borrar por id y reinsertar)
try:
    data = vs.get()
    if data and data.get("ids"):
        some_id = data["ids"][0]
        print("Borrando id:", some_id)
        vs.delete([some_id])
        # Reinsertar un documento corto como ejemplo
        from langchain_core.documents import Document
        vs.add_documents([Document(page_content="Documento de prueba actualizado", metadata={"source": base_source, "page": 999})])
        vs.persist()
        print("Actualización simulada realizada (delete + add).")
except Exception as e:
    print("[Aviso] No se pudo actualizar/borrar:", e)

# Cerrar y reabrir la colección (verificar persistencia)
vs = None
vs = Chroma(collection_name=COLLECTION_NAME, persist_directory=PERSIST_DIR, embedding_function=embeddings)
info = vs.get()
print("Reabierta colección desde disco. Total embeddings:", len(info.get("ids", [])))

## Alternativas de BD vectorial: FAISS y Qdrant
Construimos índices FAISS y Qdrant (en memoria) con los mismos embeddings y comparamos una búsqueda.

In [ ]:
# FAISS y Qdrant: construcción y búsqueda
try:
    with timer() as t_f:
        faiss_vs = FAISS.from_documents(chunks, embeddings)
    with timer() as t_fr:
        faiss_docs = faiss_vs.similarity_search("¿Qué es RAG?", k=4)
    print(f"FAISS -> index {t_f.dt:.2f}s, search {t_fr.dt:.2f}s, docs: {len(faiss_docs)}")
except Exception as e:
    print("[Aviso] FAISS no disponible:", e)

if Qdrant is not None:
    try:
        client = qdrant_client.QdrantClient(location=":memory:")
        with timer() as t_q:
            qdr_vs = Qdrant.from_documents(chunks, embeddings, client=client, collection_name="docs_x_mem")
        with timer() as t_qr:
            qdr_docs = qdr_vs.similarity_search("¿Qué es RAG?", k=4)
        print(f"Qdrant -> index {t_q.dt:.2f}s, search {t_qr.dt:.2f}s, docs: {len(qdr_docs)}")
    except Exception as e:
        print("[Aviso] Qdrant no disponible:", e)
else:
    print("[Nota] Qdrant no instalado; omitiendo ejemplo.")

print("\nPros/Contras (resumen):\n- FAISS: rápido, local, sin servidor; no metadatos avanzados por defecto.\n- Qdrant: servidor/embebido, filtros ricos, escalable; requiere cliente/servicio.\n- Chroma: sencillo, persistencia local fácil, filtros; ideal para prototipos.")

## Refinamiento de prompts: variantes y control de estilo
Creamos varias plantillas (base, breve, pasos numerados, abstención estricta) y permitimos elegir al preguntar.

In [ ]:
# Variantes de prompt
def make_prompt(variant: str = "base"):
    if variant == "breve":
        sys = ("Eres conciso. Responde en español usando SOLO el CONTEXTO. "
               "Máximo 5 líneas. Añade 'Fuentes:' al final.")
    elif variant == "pasos":
        sys = ("Eres didáctico. Responde en español en pasos numerados usando SOLO el CONTEXTO. "
               "Si no hay evidencia, indícalo. Añade 'Fuentes:' al final.")
    elif variant == "estricto":
        sys = ("Eres riguroso. NO inventes. Responde en español solo si la evidencia en CONTEXTO es suficiente. "
               "Si no, responde 'No hay evidencia suficiente'. Añade 'Fuentes:' al final.")
    else:
        sys = ("Eres un asistente experto. Responde en español usando SOLO el CONTEXTO. "
               "Incluye 'Fuentes:' al final con source y page.")
    return ChatPromptTemplate.from_messages([
        ("system", sys),
        ("human", "Pregunta: {question}\n\nCONTEXTO:\n{context}\n")
    ])


def enforce_sources(answer: str, docs: List[Any]) -> str:
    # Asegurar sección Fuentes con deduplicación
    lines = []
    seen = set()
    for d in docs:
        m = d.metadata or {}
        src = m.get("source", "?")
        page = m.get("page", m.get("page_number", "?"))
        key = (src, page)
        if key not in seen:
            seen.add(key)
            lines.append(f"- {src} (página {page})")
    if not lines:
        cite = "Fuentes: (no encontradas)"
    else:
        cite = "Fuentes:\n" + "\n".join(lines)
    ans = answer or ""
    if "Fuentes:" not in ans:
        ans = ans.rstrip() + "\n\n" + cite
    return ans


def build_rag_chain_with_prompt(retriever, llm, prompt_variant: str):
    pr = make_prompt(prompt_variant)
    chain = (
        {"context": retriever | RunnableLambda(format_docs),
         "question": RunnablePassthrough()}
        | pr
        | llm
        | parser
    )
    return chain


def ask2(question: str, llm, retriever, prompt_variant: str = "base"):
    chain = build_rag_chain_with_prompt(retriever, llm, prompt_variant)
    with timer() as t:
        answer = chain.invoke(question)
    docs_used = retriever.get_relevant_documents(question)
    answer = enforce_sources(answer, docs_used)
    print(f"Tiempo total: {t.dt:.2f}s | Prompt: {prompt_variant}")
    print_sources(docs_used)
    return answer, docs_used

# Ejemplo de uso de variantes
if llm_ollama is not None:
    a1, _ = ask2("Resume qué es RAG.", llm_ollama, retriever_sim, prompt_variant="breve")
    print("\nRespuesta breve:\n", a1[:800])